Here, we will build the SESTM Algorithm from scratch and give a brief description for what each part of the functions does

In [ ]:
import pandas as pd
import numpy as np 

## Helper Function - SESTM Algorithm

### Creates the dataframe for the input

In [ ]:
def prepare_sestm_data(
    data: pd.DataFrame,
    *,
    text_col: str = "all_text",
    return_col: str = "day_return",
    market_date_col: str = "market_date",
) -> pd.DataFrame:

    """
    Clean and validate the dataframe used to train SESTM.
    There are two versions for return_col, it can be three_day_return or day_return which can be one 
    of the factors that acts as the input to the train.
    """

    required = {
        text_col,
        return_col,
        market_date_col,
    }

    missing = required - set(data.columns)

    if missing:
        raise KeyError(
            f"Missing columns: {sorted(missing)}"
        )

    result = data.copy()

    result[text_col] = (
        result[text_col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    result[return_col] = pd.to_numeric(
        result[return_col],
        errors="coerce",
    )

    result[market_date_col] = pd.to_datetime(
        result[market_date_col],
        errors="coerce",
    ).dt.normalize()

    result = result[
        result[text_col].ne("")
        & result[return_col].notna()
        & result[market_date_col].notna()
    ].copy()

    result = result.sort_values(
        market_date_col
    ).reset_index(drop=True)

    return result

### Creates a MxN Matrix that acts as a word counter/presence counter

In [ ]:
from collections.abc import Collection

from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer


def build_word_matrices(
    data: pd.DataFrame,
    *,
    text_col: str = "all_text",
    min_document_count: int = 50,
    max_document_fraction: float = 0.98,
    stop_words: Collection[str] | None = None,
):
    
    """
    Build two matrices:

    word_counts:
        Number of times each word appears in each article.

    word_presence:
        1 when an article contains the word, otherwise 0.

    We can set the parameter min_document_count depending on the size of training set
    """

    vectorizer = CountVectorizer(
        lowercase=True,
        token_pattern=r"(?u)\b[^\W\d_]{2,}\b",
        ngram_range=(1, 1),
        min_df=min_document_count,
        max_df=max_document_fraction,
        stop_words=stop_words,
        dtype=np.float64,
    )

    word_counts = vectorizer.fit_transform(
        data[text_col]
    )

    if word_counts.shape[1] == 0:
        raise ValueError(
            "No words remain after applying "
            "min_document_count and max_document_fraction."
        )

    words = np.asarray(
        vectorizer.get_feature_names_out()
    )

    word_presence = word_counts.copy()

    word_presence.data = np.ones(
        word_presence.nnz,
        dtype=np.float64,
    )

    return {
        "vectorizer": vectorizer,
        "words": words,
        "word_counts": word_counts,
        "word_presence": word_presence,
    }

### Screening Algorithm to see most positive/negatively linked words

In [ ]:
def screen_sentiment_words(
    word_presence,
    returns: np.ndarray,
    words: np.ndarray,
    *,
    n_positive_words: int = 100,
    n_negative_words: int = 100,
):
    """
    Select the words most strongly linked to positive and
    negative return signs.

    Change n_pos/n_neg words if u want a larger or smaller set of them
    """

    returns = np.asarray(
        returns,
        dtype=float,
    )

    positive_mask = returns > 0

    overall_positive_rate = (
        positive_mask.mean()
    )

    if overall_positive_rate == 0:
        raise ValueError(
            "No positive returns exist in the training data."
        )

    if overall_positive_rate == 1:
        raise ValueError(
            "No non-positive returns exist in the training data."
        )

    # Number of articles containing each word
    document_count = np.asarray(
        word_presence.sum(axis=0)
    ).ravel()

    # Number of positive-return articles containing each word
    positive_document_count = np.asarray(
        word_presence[
            positive_mask
        ].sum(axis=0)
    ).ravel()

    word_positive_rate = np.divide(
        positive_document_count,
        document_count,
        out=np.zeros_like(
            positive_document_count,
            dtype=float,
        ),
        where=document_count > 0,
    )

    screening_score = (
        word_positive_rate
        - overall_positive_rate
    )

    positive_candidates = np.flatnonzero(
        screening_score > 0
    )

    negative_candidates = np.flatnonzero(
        screening_score < 0
    )

    positive_count = min(
        n_positive_words,
        len(positive_candidates),
    )

    negative_count = min(
        n_negative_words,
        len(negative_candidates),
    )

    positive_indices = positive_candidates[
        np.argsort(
            screening_score[
                positive_candidates
            ]
        )[::-1][:positive_count]
    ]

    negative_indices = negative_candidates[
        np.argsort(
            screening_score[
                negative_candidates
            ]
        )[:negative_count]
    ]

    selected_indices = np.concatenate(
        [
            positive_indices,
            negative_indices,
        ]
    )

    screening_table = pd.DataFrame(
        {
            "vocabulary_index": np.arange(
                len(words)
            ),
            "word": words,
            "document_count": document_count,
            "positive_document_count": (
                positive_document_count
            ),
            "positive_article_rate": (
                word_positive_rate
            ),
            "screening_score": screening_score,
            
        }
    )

    return {
        "selected_indices": selected_indices,
        "positive_indices": positive_indices,
        "negative_indices": negative_indices,
        "overall_positive_rate": (
            overall_positive_rate
        ),
        "screening_table": screening_table,
    }

### Ranking Algorithm for Returns

In [ ]:
def calculate_return_ranks(
    returns: pd.Series | np.ndarray,
) -> np.ndarray:
    """
    Rank returns from low to high and scale ranks to 0–1.
    """

    return (
        pd.Series(
            returns,
            dtype=float,
        )
        .rank(
            method="average",
            pct=True,
        )
        .to_numpy(dtype=float)
    )

### Selecting Articles From TEST set with given words

In [ ]:
def build_article_word_shares(
    word_counts,
    selected_indices: np.ndarray,
):

    ''' 
    This function is used by the predictor function to check which articles can be scored,
    basically its the articles that will be scored
    '''
    selected_counts = word_counts[
        :,
        selected_indices,
    ].tocsr()

    selected_totals = np.asarray(
        selected_counts.sum(axis=1)
    ).ravel()

    usable_articles = selected_totals > 0

    if usable_articles.sum() == 0:
        raise ValueError(
            "No article contains any selected word."
        )

    usable_counts = selected_counts[
        usable_articles
    ]

    usable_totals = selected_totals[
        usable_articles
    ]

    article_word_shares = (
        sparse.diags(
            1.0 / usable_totals
        )
        @ usable_counts
    )

    return {
        "selected_counts": selected_counts,
        "selected_totals": selected_totals,
        "usable_articles": usable_articles,
        "article_word_shares": article_word_shares,
    }

### Scoring sentiments with SESTM Method

In [ ]:
def estimate_sentiment_profiles(
    article_word_shares,
    usable_return_ranks: np.ndarray,
):
    
    ''' 
    Given the article words share and the returns ranked,  
    we will output the negative/positive word profile weights
    each weights for positive/negative adds up to 1
    '''

    rank_weights = np.column_stack(
        [
            usable_return_ranks,
            1.0 - usable_return_ranks,
        ]
    )

    left_part = np.asarray(
        article_word_shares.T
        @ rank_weights
    )

    right_part = np.linalg.pinv(
        rank_weights.T
        @ rank_weights
    )

    profiles = left_part @ right_part

    profiles = np.clip(
        profiles,
        a_min=0,
        a_max=None,
    )

    profile_sums = profiles.sum(
        axis=0,
        keepdims=True,
    )

    if np.any(profile_sums == 0):
        raise ValueError(
            "A fitted profile contains no positive values."
        )

    profiles = profiles / profile_sums

    positive_profile = profiles[:, 0]
    negative_profile = profiles[:, 1]

    return {
        "positive_profile": positive_profile,
        "negative_profile": negative_profile,
        "rank_weights": rank_weights,
    }

### Sorting the Sentiment Scores per Day

In [5]:
def aggregate_stock_day_sentiment(
    scored_data: pd.DataFrame,
    *,
    date_col: str = "market_date",
    stock_col: str = "stock",
    score_col: str = "sestm_score",
    return_col: str = "next_day_return",
) -> pd.DataFrame:

    result = scored_data.copy()

    result[date_col] = pd.to_datetime(
        result[date_col],
        errors="coerce",
    ).dt.normalize()

    # Sanity check:
    check = (
        result
        .groupby([date_col, stock_col])[return_col]
        .nunique(dropna=True)
    )

    if (check > 1).any():
        raise ValueError(
            "Same stock/date has multiple next_day_return values."
        )

    stock_day = (
        result
        .dropna(
            subset=[
                date_col,
                stock_col,
                score_col,
            ]
        )
        .groupby(
            [date_col, stock_col],
            as_index=False,
        )
        .agg(
            sentiment_score=(
                score_col,
                "mean",
            ),
            article_count=(
                score_col,
                "size",
            ),
            next_day_return=(
                return_col,
                "first",
            ),
        )
    )

    stock_day["sentiment_rank"] = (
        stock_day
        .groupby(date_col)["sentiment_score"]
        .rank(
            method="average",
            pct=True,
        )
    )

    return stock_day

## Main Model and Features

### DataClass - the output of the model

In [ ]:
from dataclasses import dataclass
from typing import Any


@dataclass # can be called with dataclass.vectorizer / etc
class SESTMModel:
    vectorizer: CountVectorizer
    selected_indices: np.ndarray
    selected_words: np.ndarray
    positive_profile: np.ndarray
    negative_profile: np.ndarray
    average_rank: float
    overall_positive_rate: float
    word_table: pd.DataFrame
    parameters: dict[str, Any]

### Main Training Function : Parameters

In [3]:
def train_sestm(
    train_data: pd.DataFrame,
    *,
    text_col: str = "all_text",
    return_col: str = "day_return",
    market_date_col: str = "market_date",
    n_positive_words: int = 100,
    n_negative_words: int = 100,
    min_document_count: int = 50,
    max_document_fraction: float = 0.98,
    stop_words: Collection[str] | None = None,
) -> SESTMModel:
    
    """
    Train SESTM from article text and linked stock returns.
    Returns the parameters use for the prediction
    """

    # 1. Clean rows
    data = prepare_sestm_data(
        train_data,
        text_col=text_col,
        return_col=return_col,
        market_date_col=market_date_col,
    )

    if len(data) < 100:
        raise ValueError(
            "Too few valid training rows."
        )

    # 2. Build word-count and word-presence matrices
    word_data = build_word_matrices(
        data,
        text_col=text_col,
        min_document_count=(
            min_document_count
        ),
        max_document_fraction=(
            max_document_fraction
        ),
        stop_words=stop_words,
    )

    returns = data[
        return_col
    ].to_numpy(dtype=float)

    # 3. Screen words
    screening = screen_sentiment_words(
        word_data["word_presence"],
        returns,
        word_data["words"],
        n_positive_words=(
            n_positive_words
        ),
        n_negative_words=(
            n_negative_words
        ),
    )

    selected_indices = screening[
        "selected_indices"
    ]

    selected_words = word_data[
        "words"
    ][selected_indices]

    # 4. Rank returns
    return_ranks = calculate_return_ranks(
        returns
    )

    # 5. Calculate article word shares
    share_data = build_article_word_shares(
        word_data["word_counts"],
        selected_indices,
    )

    usable_articles = share_data[
        "usable_articles"
    ]

    usable_return_ranks = return_ranks[
        usable_articles
    ]

    # 6. Estimate positive and negative profiles
    profiles = estimate_sentiment_profiles(
        share_data["article_word_shares"],
        usable_return_ranks,
    )

    positive_profile = profiles[
        "positive_profile"
    ]

    negative_profile = profiles[
        "negative_profile"
    ]

    # Build a readable word table
    screening_table = (
        screening["screening_table"]
        .set_index("vocabulary_index")
        .loc[selected_indices]
        .reset_index()
    )

    word_table = screening_table.copy()

    word_table["positive_profile"] = (
        positive_profile
    )

    word_table["negative_profile"] = (
        negative_profile
    )

    word_table["tone"] = (
        positive_profile
        - negative_profile
    )

    word_table["screening_side"] = np.where(
        word_table["screening_score"] > 0,
        "positive",
        "negative",
    )

    return SESTMModel(
        vectorizer=word_data["vectorizer"],
        selected_indices=selected_indices,
        selected_words=selected_words,
        positive_profile=positive_profile,
        negative_profile=negative_profile,
        average_rank=float(
            return_ranks.mean()
        ),
        overall_positive_rate=float(
            screening[
                "overall_positive_rate"
            ]
        ),
        word_table=word_table,
        parameters={
            "n_positive_words": (
                n_positive_words
            ),
            "n_negative_words": (
                n_negative_words
            ),
            "min_document_count": (
                min_document_count
            ),
            "max_document_fraction": (
                max_document_fraction
            ),
            "training_rows": len(data),
            "usable_training_rows": int(
                usable_articles.sum()
            ),
        },
    )

### Predicting Function : Scoring Sentiment 

In [4]:
from scipy.optimize import minimize_scalar

def score_sestm_articles(
    model: SESTMModel,
    data: pd.DataFrame,
    *,
    text_col: str = "all_text",
    output_col: str = "sestm_score",
    penalty_strength: float = 10.0,
) -> pd.DataFrame:
    """
    Score new articles using a trained SESTM model.
    """

    if text_col not in data.columns:
        raise KeyError(
            f"Missing text column: {text_col}"
        )

    result = data.copy()

    texts = (
        result[text_col]
        .fillna("")
        .astype(str)
    )

    full_counts = model.vectorizer.transform(
        texts
    )

    selected_counts = full_counts[
        :,
        model.selected_indices,
    ].tocsr()

    scores = np.full(
        len(result),
        model.average_rank,
        dtype=float,
    )

    for row_position in range(
        selected_counts.shape[0]
    ):
        counts = (
            selected_counts
            .getrow(row_position)
            .toarray()
            .ravel()
        )

        total_count = counts.sum()

        # No selected sentiment words
        if total_count == 0:
            continue

        def negative_objective(
            p: float,
        ) -> float:
            p = np.clip(
                p,
                1e-8,
                1.0 - 1e-8,
            )

            mixed_profile = (
                p
                * model.positive_profile
                + (1.0 - p)
                * model.negative_profile
            )

            mixed_profile = np.clip(
                mixed_profile,
                1e-12,
                None,
            )

            average_log_likelihood = (
                counts
                @ np.log(mixed_profile)
            ) / total_count

            penalty = (
                2.0
                * penalty_strength
                * (
                    model.average_rank
                    * np.log(p)
                    + (
                        1.0
                        - model.average_rank
                    )
                    * np.log(1.0 - p)
                )
            )

            objective = (
                average_log_likelihood
                + penalty
            )

            # scipy minimizes, so reverse the sign.
            return -objective

        optimization = minimize_scalar(
            negative_objective,
            bounds=(
                1e-6,
                1.0 - 1e-6,
            ),
            method="bounded",
        )

        scores[row_position] = (
            optimization.x
        )

    result[output_col] = scores

    return result